In [0]:
storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

sql_server = "sql-edgar-analytics.database.windows.net"
sql_database = "edgar-analytics"
sql_user = "edgaradmin"
sql_password = dbutils.secrets.get("kalshi-scope", "sql-password")

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};user={sql_user};password={sql_password};encrypt=true;trustServerCertificate=false"

print("Ready")

In [0]:
from pyspark.sql.functions import col, when, left, current_timestamp, round as spark_round, to_date, datediff, lit, month, year, weekofyear, current_date
from pyspark.sql.functions import substring

df_silver = spark.read.format("delta").load(f"{silver_path}/kalshi_markets")

def map_category(prefix_col):
    return (
        when(prefix_col.isin("KXNFLD","KXNFLP","KXNFLT","KXNFLO","KXNFLN","KXNFLA","KXNFLM","KXNFLN"), "NFL")
        .when(prefix_col.isin("KXNBA1","KXNBA2","KXNBA3","KXNBAP","KXNBAR","KXNBAS","KXNBAT","KXNBAW","KXNBAA","KXNBAC","KXNBAD","KXNBAE","KXNBAF","KXNBAG","KXNBAM","KXNBAN","KXNBAB","KXNBA-"), "NBA")
        .when(prefix_col.isin("KXMLBH","KXMLBT","KXMLBS","KXMLBW","KXMLBN","KXMLBA","KXMLBM","KXMLBF","KXMLBK","KXMLBR","KXMLBE","KXMLBL","KXMLBP","KXMLBG","KXMLB-","KXMLBB"), "MLB")
        .when(prefix_col.isin("KXNCAA"), "NCAA")
        .when(prefix_col.isin("KXNHLT","KXNHLS","KXNHLP","KXNHLR","KXNHLA","KXNHLC","KXNHLG","KXNHLH","KXNHLV","KXNHLW","KXNHLE","KXNHLM","KXNHLN","KXNHL-"), "NHL")
        .when(prefix_col.isin("KXPGAT","KXPGAR","KXPGAM","KXPGAH","KXLPGA","KXPGAC"), "Golf")
        .when(prefix_col.isin("KXATPS","KXATPC","KXATPM","KXATPG","KXATP1","KXWTAM","KXWTAG","KXWTIM","KXWTAS"), "Tennis")
        .when(prefix_col.isin("KXCS2M","KXCS2G","KXCS2T","KXVALO","KXLOLG","KXLOLM","KXLOLT","KXR6GA"), "Esports")
        .when(prefix_col.isin("KXUFCF","KXUFCL","KXUFCW","KXUFCB","KXUFCH","KXUFCM"), "UFC/MMA")
        .when(prefix_col.isin("KXSERI","KXBUND","KXLIGU","KXEPL1","KXEPLG","KXEPLR","KXEPLT","KXEPLS","KXEPLB","KXUCL","KXUECL","KXWCRO","KXWCGA","KXWCGR","KXWCGO","KXWCSQ","KXFIFA"), "Soccer")
        .when(prefix_col.isin("KXSENA","KXTRUM","KXPRES","KXVOTE","KXBILL","KXCONG","KXVPRE","SENATE","HOUSEC","HOUSEI","HOUSEM","HOUSEN","HOUSEO","HOUSEP","HOUSEV","HOUSEW","HOUSEF","HOUSEA","HOUSEI","KXGOVM","KXGOVA","KXGOVC","KXGOVN","KXGOVT","KXGOVI","KXGOVK","KXGOVV","KXGOVW","KXGOVG","KXGOVP","KXGOVR","KXGOVS","KXGOVO","KXGOVF","KXPRIM","KXGAPR","KXKYPR","KXORPR","KXALPR","KXIDPR","KXPAPR","KXLAPR","KXMDPR","KXNVPR","KXOHPR","KXTXPR","KXWVPR","KXBRPR","GOVPAR"), "Politics/Government")
        .when(prefix_col.isin("KXECON","KXFED-","KXFEDD","KXFEDM","KXFEDE","KXFEDC","KXFEDG","KXFEDL","KXFOMC","KXRATE","KXCPI-","KXCPIC","KXCPIY","KXGDPN","KXGDPY","KXGDP-","KXUSTY","KXADP-","KXPAYR","KXJOBL","KXNASD","KXNASC","KXINX-","KXINXU","KXINXY","KXINXM","KXINXP","KXSP50","KXBOND","KXEOWE","KXEOTR"), "Economics/Markets")
        .when(prefix_col.isin("KXBTC-","KXBTCD","KXBTCM","KXBTCY","KXBTCH","KXBTC1","KXBTC2","KXBTCR","KXBTCV","KXETH-","KXETHD","KXETHM","KXETHY","KXETH1","KXDOGE","KXBNB-","KXBNBD","KXXRP-","KXXRPD","KXXRPM","KXXRP1","KXSHIB","KXSOL1","KXSOL2","KXSOLM","KXDASH","KXCRYP","KXTOKE"), "Crypto")
        .when(prefix_col.isin("KXGOLD","KXSILV","KXCOPP","KXNATG","KXWTI-","KXWTIW","KXOILR","KXNGAS"), "Commodities")
        .when(prefix_col.isin("KXOSCA","KXGRAM","KXSONG","KXTAYL","KXMOVI","KXSNLH","KXSNLM","KXSNL-","KXPODC","KXTVSH","KXBIGB","KXSURV","KXACTO","KXSONG","KXGAME","KXROLL"), "Entertainment")
        .when(prefix_col.isin("KXMUSK","KXTESL","KXMETA","KXAISP","KXAIST","KXLLM1","KXGPTC","KXAAPL","KXSPAC"), "Tech/AI")
        .when(prefix_col.isin("KXTEMP","KXRAIN","KXTORN"), "Weather")
        .when(prefix_col.isin("KXBRAS","KXEURO","KXFIFA","KXWCRO","KXINTL","KXPUTI","KXISRA","KXUKRE","KXIRAN"), "International")
        .when(prefix_col.isin("KXTRUT","KXTRUTHSOCIAL"), "Trump/Social Media")
        .when(prefix_col.isin("KXWNBA"), "WNBA")
        .when(prefix_col.isin("KXF1-2","KXF1CO","KXF1CH","KXF1OC"), "Formula 1")
        .when(prefix_col.isin("KXAFLG"), "Australian Football")
        .when(prefix_col.isin("KXFIBA","KXWBCF","KXWBCB","KXWBCM","KXWBCW","KXWBCH","KXWBCL"), "Other Sports")
        .when(prefix_col.isin("KXUE-A","KXUE-B","KXUE-C","KXUE-F","KXUE-G","KXUE-I","KXUE-J","KXUE-M","KXUE-R","KXUE-U","KXUEL-","KXUECL"), "Soccer")
        .when(prefix_col.isin("KXNETF","KXAAAG","KXRTX5","KXH100","KXH200","KXB200","KXA100"), "ETFs/Stocks")
        .when(prefix_col.isin("KXIPOO","KXIPOA","KXIPOB","KXIPOC","KXIPOD","KXIPOF","KXIPOG","KXIPOR","KXIPOS","KXIPOW","KXIPO-"), "IPOs")
        .when(prefix_col.isin("KXHOUS"), "Housing/Real Estate")
        .when(prefix_col.isin("KXMUSK","KXTESL","KXMETA","KXAISP","KXAIST","KXLLM1","KXGPTC","KXAAPL","KXSPAC","KXGROK","KXJPMO","KXJPMC"), "Tech/AI")
        .when(prefix_col.isin("KXNEXT","KXLEAD","KXRANK","KXTEAM","KXHYPE","KXROLE","KXSOLD","KXSOLE","KXHIGH","KXLALI","KXSTAR","KXTOPA","KXTOPS","KXTOPM","KXTOPC","KXTOPB","KXTOP3","KXTOPP","KX1SON","KX10SO","KX20SO","KX1ALB","KXPERF","KXSPOT","KXRECO","KXLOSE"), "Pop Culture/Rankings")
        .when(prefix_col.isin("KXALBU","KXSONG","KXGRAM","KXOSCA","KXMOVI","KXSNLH","KXSNLM","KXSNL-","KXPODC","KXTVSH","KXBIGB","KXSURV","KXACTO","KXROLL","KXVENU","KXMENW","KXGAME","KXSUPE","KXBREN","KXFEAT","KXLOVE","BEYONC","KXSWIF","KXTAYL"), "Entertainment")
        .when(prefix_col.isin("KXLOWT","KXCBDE","KXMEDI","KXPCEC","KXSECP","KXSCOU","KXATTY","KXFDAA","KXFEDC","KXFEDE","KXFEDG","KXFEDL"), "Policy/Legal")
        .when(prefix_col.isin("KXBRAS","KXEURO","KXINTL","KXPUTI","KXISRA","KXUKRE","KXIRAN","KXVENE","KXBRAZ","KXTHAI","KXMARM","KXTURK","KXGREE","KXBELG","KXDENM","KXFINL","KXSPAI","KXSWED","KXITAL","KXPOLA","KXNIGE","KXMALA","KXKENY","KXGHAN","KXZAMB","KXAFRI","KXLATV","KXPERU","KXARGE","KXMONG","KXMEXC","KXMOLD","KXBULG","KXSLOV","KXICER","KXDOMI","KXCANA","KXCOLO","KXQUEB","KXWALE","KXSCOT","KXNEWZ","KXAUST","KXJAPA","KXSEOU","KXOAIA"), "International")
        .when(prefix_col.isin("KXRT-D","KXRT-M","KXRT-S","KXRT-Y","KXRT-A","KXRT-O"), "Real-Time Markets")
        .when(prefix_col.isin("KXCBAG","KXSB-2","KXBALL","KXAMER","KXTOUR","KXRUGB","KXBOXL","KXBOXI","KXSOCC","KXINDY","KXMOTO","KXU3MA","KXU3-2","KXUCLS","KXUCLT","KXUCLG","KXUCLW","KXUCL1","KXUCL-","KXUCLR","KXUCLB","KXUCLF"), "Other Sports")
        .otherwise("Other")
    )

prefix = substring(col("event_ticker"), 1, 6)

df_gold = df_silver.withColumn("prefix", prefix) \
    .withColumn("category", map_category(col("prefix"))) \
    .withColumn("implied_prob_yes", spark_round((col("yes_ask") + col("yes_bid")) / 2, 4)) \
    .withColumn("implied_prob_no", spark_round((col("no_ask") + col("no_bid")) / 2, 4)) \
    .withColumn("spread", spark_round(col("yes_ask") - col("yes_bid"), 4)) \
    .withColumn("is_single_market", col("mve_collection_ticker").isNull()) \
    .withColumn("has_liquidity", col("liquidity") > 0) \
    .withColumn("market_quality",
        when(col("liquidity") > 1000, "High")
        .when(col("liquidity") > 100, "Medium")
        .when(col("liquidity") > 0, "Low")
        .otherwise("None")
    ) \
    .withColumn("close_date", to_date(col("close_time"))) \
    .withColumn("days_until_close", datediff(col("close_date"), current_date())) \
    .withColumn("close_month", month(col("close_time"))) \
    .withColumn("close_year", year(col("close_time"))) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .select(
        "ticker",
        "title",
        "event_ticker",
        "category",
        "status",
        "yes_ask",
        "yes_bid",
        "no_ask",
        "no_bid",
        "last_price",
        "implied_prob_yes",
        "implied_prob_no",
        "spread",
        "liquidity",
        "volume_24h",
        "volume_total",
        "open_interest",
        "market_quality",
        "is_single_market",
        "has_liquidity",
        "close_time",
        "close_date",
        "days_until_close",
        "close_month",
        "close_year",
        "is_provisional",
        "mve_collection_ticker",
        "no_sub_title",
        "yes_sub_title",
        "result",
        "can_close_early",
        "gold_updated_at"
    )

print(f"Gold row count: {df_gold.count()}")
df_gold.groupBy("category").count().orderBy("count", ascending=False).show(30, truncate=False)

In [0]:
(df_gold.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "kalshi_gold.fact_markets")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("overwrite")
    .save()
)

print("Gold table written to Azure SQL successfully")